# Why are small files a problem?

Spark does file-level Planning. For each file, it reads metadata and creates at least one task
If we have 10GB of data with 10,000 1 MB files

For this, Spark must:
1. List 1M files from storage (S3 / HDFS)
2. Create 1M tasks (number of files = number of tasks is not always True. Spark combines multiple small files, but sometimes depending on file type and compression, it does not combine them)
3. Hold file metadata in Driver memory

Result
1. Slow job start
2. Driver OOM 
3. High NameNode / S3 LIST latency

Also, each Spark task has overhead:
1. JVM startup
2. Serialisation
3. Network coordination

# Solutions

## 1. Control Number of Output Files
The best solution is not have a large number of small files. 
Use 
1. df.coalesce(50).write.parquet(path) # This operation is cheap because no shuffle
2. df.repartition(50).write.parquet(path) # This shuffles the data, use when fixing skew
3. Proper Partitioning Strategy. Instead of year/month/day/hour, reduce to year/month/day or year/month.4
4. Delta / Iceberg / Hudi Merge Files 


## 2. Executor & Memory Configuration

### Executor sizing (Golden Rule)
1. spark.executor.cores=4
2. spark.executor.memory=16g
3. spark.executor.memoryOverhead=4g
==> Avoids GC storms, Prevents OOM on shuffle-heavy jobs

### Driver
    * Driver holds file metadata & DAG
    * Small files = large driver pressure
1. spark.driver.memory=8g
2. spark.driver.maxResultSize=2g



# 3. Handle during read
1. Handle a large number of partitions.
    1. spark.sql.files.maxPartitionBytes configuration decides the maximum number of bytes to pack into a single partition when reading from file-based data sources. By default, it combines small files into one partition.
    2. If partitions are not combined, use coalesce or repartitions.<br><br>


2. Handle Metadata
    1. Increase resources for the driver and executor.
    2. write intermidiate result of combined files into large files and create the dataframe again.

In [1]:
import os
import pandas as pd
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col



output_dir = "small_files_data"
Path(output_dir).mkdir(exist_ok=True)

num_files = 10000
rows_per_file = 1  

for i in range(num_files):
    df = pd.DataFrame({
        "id": [i],
        "value": [f"value_{i}"]
    })
    df.to_csv(f"{output_dir}/file_{i}.csv", index=False)

print(f"Created {num_files} small CSV files")


Created 10000 small CSV files


In [2]:

spark = SparkSession.builder \
    .appName("SmallFilesExample") \
    .getOrCreate()

df = spark.read.csv(output_dir, header=True)
print("Number of partitions:", df.rdd.getNumPartitions())
df.show(5)


print("Total rows:", df.count())

print("Partitions before repartition:", df.rdd.getNumPartitions())

Number of partitions: 313
+----+----------+
|  id|     value|
+----+----------+
|1000|value_1000|
|1001|value_1001|
|1002|value_1002|
|1003|value_1003|
|1004|value_1004|
+----+----------+
only showing top 5 rows
Total rows: 10000
Partitions before repartition: 313


Right now, my Docker image is using 2GB of RAM.<br>
Let's coalesce it and write it to some bigger files, and check the size again

In [3]:

df_repartition = df.repartition(10)  # merge into 10 partitions

print("Partitions after coalesce:", df_repartition.rdd.getNumPartitions())
df_repartition.show(5)

Partitions after coalesce: 10
+----+----------+
|  id|     value|
+----+----------+
|1000|value_1000|
|1026|value_1026|
|1012|value_1012|
|1007|value_1007|
|1060|value_1060|
+----+----------+
only showing top 5 rows


In [4]:
output_compact = "compact_files_data"
df_repartition.write.mode("overwrite").parquet(output_compact)

print(f"Data written compactly to {output_compact}")


Data written compactly to compact_files_data


In [5]:
output_compact = "compact_files_data"
df_repartition = spark.read.parquet(output_compact)

In [6]:
df_repartition.rdd.getNumPartitions()

10

In [7]:
spark.stop()

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SmallFilesExample") \
    .getOrCreate()

output_compact = "compact_files_data"
df = spark.read.parquet(output_compact)